About Cryptography

- 대수학은 일방향 함수들로 가득한 보물창고이다!
    - NP-complete 문제를 활용한 암호시스템($P \neq NP$)를 가정함으로서 안정성을 담보 받음
        - TSP, 해밀턴 사이클, Knapsack, 최장거리 경로 등
    - 대부분 이산로그 문제들은 일방향 mapping이 이루어지는 구조체에서 발견됨!
        - 타원 곡선 유한체, 소인수분해(shor는 O((logN)^3)으로 양자 gate는 더 넓은 범위의 상상력 ㄱㄴ)
- 결정론적인 시스템(평문-암호문 일대일 대응)의 해독성에서 비롯된 확률적인 암호시스템(polly cracker)의 대두

Branch

- Symmetry-Key(use same key) / Assymetric-Key(use different key for enc/dec)
    - ZKP / Homomorphism Encryption / ...
- Steganography
- Quantum Cryptography


Details

[PGP](namu.wiki/w/PGP) (Pretty Good Privacy)

- Bob wants to send msg to Alice
- Encrypt with Alice Public Key / Sign with Bob Private Key

> ActiveX history begins with PGP / TLS + more safety

In [117]:
import random
import numpy as np
import math
import codecs

### Merkle-Hellman Knapsack System: Using NP-complete problem to form one-way function.
# Knapsack is NP-complete!
#   S = {s1, s2, ..., sn} and objective sum T, find subset S' such satisfying sum S'i = T.
#   -> with super-increasing sequenece

# Componenets
a = [] # super increasing sequence
noises = random.sample(list(range(10))*8, 20)
for i, noise in enumerate(noises):
    a.append(sum(a[:i])+noise)
m = a[-1] + random.randint(1, 10) # set modulus
w = 0 # power
for w in range(2, m):
    if math.gcd(w, m) == 1:
        break
w_inverse = 0
for w_inverse in range(m):
    if w * w_inverse % m == 1:
        break
public_key = list(map(lambda x: (x * w) % m, a))

make_binary = lambda string: bin(int(string.encode("utf-8").hex(), 16))[2:]
data = make_binary("HI")

data = [int(b) for b in data]
encoded = sum([(public_key[i] * b) % m for i, b in enumerate(data)])
print(f"Encode msg 'HI' into {encoded} with {','.join(map(str, public_key))}")

# w has its inverse, we could get data * w * a -> data * a (multiflying w^-1)
# in attacker perspective, factor C = sum(xi * b) -> NP-complete
decoded = (encoded * w_inverse) % m
decoded_data = []
for ai in reversed(a):
    if decoded >= ai:
        decoded -= ai
        decoded_data.append(1)
    else: decoded_data.append(0)
decoded_data = decoded_data[::-1]

binary_to_string = lambda binary: codecs.decode(hex(int("".join(map(str, binary)), 2))[2:], "hex").decode('utf-8')
print(f"Decoded {binary_to_string(decoded_data[:len(data)])}")

Encode msg 'HI' into 263406 with 15,24,63,111,228,444,903,1806,3600,7215,14436,28851,57705,115404,230829,461658,923301,1846593,1231055,2462128
Decoded HI


In [124]:
### Lattice Reduction Attack -> Merkle-Hellman is not safe!
# LLL finds the short Latice Vector to break it! in Polynomial time.

# BX = C (lattice perspective)
# 이 lattice에서 LLL 알고리즘을 통해 "짧은" 벡터 X를 찾자!

# Process
# - 1. Gram-Schmidt Process
# - 2. Size Reduction
# - 3. Lovasz condition check and Swap

b = np.diag(public_key+[1], k=0).astype(np.float64)
b[:-1, -1] = -encoded 

# Gram-Schmidt Process
def gramschmidt(b):
    bstar = np.zeros_like(b)
    for i in range(len(b)):
        bstar_i = b[i]
        for j in range(i):
            bstar_j = bstar[j]
            mu_ij = bstar_j @ bstar_i / (bstar_j @ bstar_j)
            bstar_i -= mu_ij * bstar_j
        bstar[i] = bstar_i
    return bstar

_ = gramschmidt(b)

In [105]:
### ECDSA, who enables Digital Life.

from ecdsa.curves import SECP256k1
from ecdsa.ellipticcurve import Point
import hashlib

#Generator
x1 = 55066263022277343669578718895168534326250603453777594175500187360389116729240
y1 = 32670510020758816978083085130507043184471273380659243275938904335757337482424
G = Point(SECP256k1.curve, x1, y1)
print(G==G * (SECP256k1.order+1))
print("generator?", G == SECP256k1.generator)

n = SECP256k1.order

#Public key, Private key
k = 1002349230423
P = G*k

#Random point
l = 10
# r = 36322260242567644327577471914851727161017458705958127170915236715425819333073
# y = 113817104126258647026551196310596962231430747658282676626153792268748326724326
# R = Point(SECP256k1.curve, r, y)
R = G*l

#message hash
m = b"Don't Trust, Verify"
hash_obj = hashlib.sha256(m)
hash_hex = hash_obj.hexdigest()
z = int(hash_hex, 16)
print("message:", m)
print("message hash:", z)

#signature
s = (l+z*k) % n

#verification
print("valid signatre?")
print(l*G + z*P == s*G)

# Schnorr signature - (비밀 분산스킴)
### R + z P = s G ----> inside z, there are sign that I agree this signature. (ex. P-R-msg)
### (l1+l2+...) + z * (k1+k2+...) = s1+s2+...

True
generator? True
message: b"Don't Trust, Verify"
message hash: 112761469845056919304416565170674990599925418308225593267094103636319905743526
valid signatre?
True


In [90]:
### ZKP with discrete logarithm
# how to verify my knowledge without exposing knowledge.
# - Completeness: Complete -> "true"
# - Soundness: P(Negative) << eps(max 1/2)
# - Zero-Knowledgeness: Verifiers only attain true/false

# ali ba ba cave
# 1. Prover walks into A or B.
# 2. Verifier requests Prover to walk out A or B.
# -> A-B transition require to have secret. Trust Prover with Probability.
# -> Prior Agreement: Prover가 비밀을 알고 있음을 Verifier가 확신하되, 제3자는 그 비밀 존재 여부조차 확실히 알 수 없어야 함.
#       > 영지식성을 보장하기 위해서는 Verifer을 제외한 모든 사람이 Prover의 secret 보유 사실을 확실할 수 없어야 함.
#       > asymmetic-key과의 차이점은 verifier가 전체인가, 한 명인가이다.

# https://hyun-jeong.medium.com/h-3c3d45861ced
import random

p = 1013
secret = 324
g = 2

y = pow(g, secret) % p # public
r = random.randint(1, p)
C = pow(g, r) % p # public

if random.random() > 0.5:
    print(r)
    print(C == pow(g, r) % p)
else:
    rr = (secret+r) % (p-1)
    print(rr) # fermet's theorem
    print(C * y % p == pow(g, rr) % p)

# 왜 두 경우로 나누어 검증을 진행하는 것일까?
# -> 외부에서 지켜봤을 때 영지식성을 보장하기 위해서! / print된 r/rr 값이 무엇인지 모르고, verifier과 prover만 공유되기에!
# --------> 그것보다는 prover가 속이는 경우를 없애려는 것. C를 공유했을 때 이것이 e=1일 때를 위해 만들어진 것인지 우리는 확신할 수 없다!! so, 둘 다 테스트 해봐야 한다.
    
# “Verifier 입장에서 본 프로토콜 수행 결과(트랜스크립트)를 돌려볼 수 있는 시뮬레이터가 존재한다”
# -> 난 '검증'만 가능하지 어떻게 이를 가능하게 만드는지는 알 수 없다!

468
True


In [ ]:
# zk-SNARKs -> Non-interactive zkp?